In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64

JupyterDash.infer_jupyter_proxy_config()

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import CRUD Python module from Project One
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
username = "aacuser"
password = "SNHU1234"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an
# invalid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will return a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'], inplace=True, errors='ignore')

# DEBUG
# print(df)

# Column positions used by the map (after _id drop)
# 4 = breed, 9 = name, 13 = location_lat, 14 = location_long
LAT_COL, LONG_COL, BREED_COL, NAME_COL = 13, 14, 4, 9

# Unique identifier
UNIQUE_ID = "CS-340 Project Two - Daniel Gonzalez"

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Encode Grazioso Salvare logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display': 'none'}),
    # Logo linked to SNHU
    html.Center(
        html.A(
            href='https://www.snhu.edu',
            target='_blank',
            children=[
                html.Img(
                    src='data:image/png;base64,{}'.format(encoded_image),
                    style={'height': '150px', 'width': 'auto'}
                )
            ]
        )
    ),
    html.Center(html.B(html.H1('Grazioso Salvare Dashboard'))),
    html.Center(html.P(UNIQUE_ID, style={'fontStyle': 'italic', 'color': '#555'})),
    html.Hr(),

    # Interactive filter options (radio items)
    html.Div([
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'Water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'Disaster'},
                {'label': 'Reset', 'value': 'Reset'}
            ],
            value='Reset',
            labelStyle={'display': 'inline-block', 'margin-right': '20px'},
            inputStyle={'margin-right': '5px'}
        )
    ], style={'textAlign': 'center', 'margin': '15px'}),

    html.Hr(),

    # Interactive data table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        column_selectable=False,
        row_selectable="single",
        row_deletable=False,
        selected_columns=[],
        selected_rows=[0],
        page_action="native",
        page_current=0,
        page_size=10,
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'padding': '5px',
            'maxWidth': '180px',
            'overflow': 'hidden',
            'textOverflow': 'ellipsis'
        },
        style_header={'fontWeight': 'bold', 'backgroundColor': '#f0f0f0'}
    ),

    html.Br(),
    html.Hr(),

    # Charts side-by-side
    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(id='graph-id', className='col s12 m6', style={'width': '50%'}),
            html.Div(id='map-id', className='col s12 m6', style={'width': '50%'})
        ]
    )
])

#############################################
# Interaction Between Components / Controller
#############################################

# Filter data table based on radio selection
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    if filter_type == 'Water':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "Labrador Retriever Mix",
                "Chesapeake Bay Retriever",
                "Newfoundland"
            ]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    elif filter_type == 'Mountain':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "German Shepherd",
                "Alaskan Malamute",
                "Old English Sheepdog",
                "Siberian Husky",
                "Rottweiler"
            ]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    elif filter_type == 'Disaster':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "Doberman Pinscher",
                "German Shepherd",
                "Golden Retriever",
                "Bloodhound",
                "Rottweiler"
            ]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }
    else:  # Reset
        query = {}

    results = db.read(query)
    dff = pd.DataFrame.from_records(results)
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)
    return dff.to_dict('records')

# Pie chart of breeds (updates with table)
@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return [html.P("No data to display.")]
    dff = pd.DataFrame.from_dict(viewData)
    if 'breed' not in dff.columns:
        return [html.P("Breed column missing.")]
    return [
        dcc.Graph(
            figure=px.pie(
                dff,
                names='breed',
                title='Preferred Breeds (Current Filter)'
            )
        )
    ]

# This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in (selected_columns or [])]

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')]
)
def update_map(viewData, index):
    if viewData is None or len(viewData) == 0:
        return [html.P("No records match the current filter.")]
    dff = pd.DataFrame.from_dict(viewData)
    row = index[0] if index else 0
    if row >= len(dff):
        row = 0
    try:
        lat = float(dff.iloc[row, LAT_COL])
        lon = float(dff.iloc[row, LONG_COL])
    except (IndexError, ValueError, TypeError):
        return [html.P("Invalid location data for selected row.")]
    return [
        dl.Map(
            style={'width': '100%', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(str(dff.iloc[row, BREED_COL])),
                        dl.Popup([
                            html.H3("Animal Name"),
                            html.P(str(dff.iloc[row, NAME_COL]))
                        ])
                    ]
                )
            ]
        )
    ]

# Run app and display result in jupyterlab mode, note, if you have previously run a prior app,
# the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server(mode='jupyterlab', port=8054)
